# 田町駅周辺 ラーメン・牛丼 分布マップ (2016 vs 2026)

田町駅・三田駅周辺のラーメン屋と牛丼屋の分布を2016年と2026年で比較する。

## データソース
- **2016年**: 食べログ地図検索のスクリーンショット（PDF）から画像処理でピン座標を抽出 + 食べログ閉店ページから住所・座標を復元
- **2026年**: 食べログから直接スクレイピング（店名・評価・座標）

In [1]:
# foliumがインストールされていない場合はインストール
import subprocess, sys
try:
    import folium
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "-q"])
    import folium

from folium.plugins import MarkerCluster
import pandas as pd
import json
import math
import os

## 1. データ読み込み

In [2]:
# Google Colab: GitHubから直接データを読み込み
import io, urllib.request

GITHUB_DATA = 'https://raw.githubusercontent.com/ryo-nakajima/undergrad-io-2026-data/main/data'

def read_csv_from_github(filename):
    url = f'{GITHUB_DATA}/{filename}'
    response = urllib.request.urlopen(url)
    return pd.read_csv(io.StringIO(response.read().decode('utf-8')))

# ローカル実行時はdata/から、Colab時はGitHubから読み込み
import os
if os.path.exists('data'):
    DATA_DIR = 'data'
    ramen_2016 = pd.read_csv(os.path.join(DATA_DIR, 'ramen_2016.csv'))
    gyudon_2016 = pd.read_csv(os.path.join(DATA_DIR, 'gyudon_2016.csv'))
    ramen_2026 = pd.read_csv(os.path.join(DATA_DIR, 'ramen_2026.csv'))
    gyudon_2026 = pd.read_csv(os.path.join(DATA_DIR, 'gyudon_2026.csv'))
else:
    ramen_2016 = read_csv_from_github('ramen_2016.csv')
    gyudon_2016 = read_csv_from_github('gyudon_2016.csv')
    ramen_2026 = read_csv_from_github('ramen_2026.csv')
    gyudon_2026 = read_csv_from_github('gyudon_2026.csv')

ramen_2026 = ramen_2026.drop_duplicates(subset='name', keep='first')
gyudon_2026 = gyudon_2026.drop_duplicates(subset='name', keep='first')

print(f'2016年: ラーメン {len(ramen_2016)}店, 牛丼 {len(gyudon_2016)}店')
print(f'2026年: ラーメン {len(ramen_2026)}店, 牛丼 {len(gyudon_2026)}店')

2016年: ラーメン 27店, 牛丼 5店
2026年: ラーメン 30店, 牛丼 6店


In [ ]:
# データプレビュー（全件表示）
pd.set_option('display.max_rows', None)

print('=== 2016 ラーメン (全店) ===')
display(ramen_2016[['name','rating_2016','status']].sort_values('status'))

print('\n=== 2016 牛丼 (全店) ===')
display(gyudon_2016[['name','rating_2016','status']])

print('\n=== 2026 ラーメン (全店) ===')
cols_r26 = ['name','rating','since_2016','opened']
cols_r26 = [c for c in cols_r26 if c in ramen_2026.columns]
display(ramen_2026[cols_r26].sort_values(['since_2016','opened'], ascending=[True,True]))

print('\n=== 2026 牛丼 (全店) ===')
display(gyudon_2026[['name','rating']])

## 2. 田町駅から800m圏のフィルタリング

In [4]:
# 田町駅の座標
TAMACHI_LAT = 35.64564
TAMACHI_LNG = 139.74753
RADIUS_M = 800

def haversine(lat1, lng1, lat2, lng2):
    """2点間の距離(m)を計算"""
    R = 6371000
    dlat = math.radians(lat2 - lat1)
    dlng = math.radians(lng2 - lng1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlng/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def filter_by_radius(df, lat_col='lat', lng_col='lng', radius=RADIUS_M):
    """田町駅からradius(m)以内の店舗のみ"""
    df = df.dropna(subset=[lat_col, lng_col]).copy()
    df['dist_m'] = df.apply(lambda r: haversine(TAMACHI_LAT, TAMACHI_LNG, r[lat_col], r[lng_col]), axis=1)
    return df[df['dist_m'] <= radius].copy()

# フィルタリング
r16 = filter_by_radius(ramen_2016)
g16 = filter_by_radius(gyudon_2016)
r26 = filter_by_radius(ramen_2026)
g26 = filter_by_radius(gyudon_2026)

print(f'田町駅から{RADIUS_M}m以内:')
print(f'  2016 ラーメン: {len(r16)}店 / 牛丼: {len(g16)}店')
print(f'  2026 ラーメン: {len(r26)}店 / 牛丼: {len(g26)}店')

田町駅から800m以内:
  2016 ラーメン: 27店 / 牛丼: 5店
  2026 ラーメン: 30店 / 牛丼: 6店


## 3. インタラクティブ比較マップ

レイヤー切り替えで以下を表示/非表示にできます:
- ラーメン 2016年（オレンジ）
- ラーメン 2026年（赤）
- 牛丼 2016年（青）
- 牛丼 2026年（緑）

In [5]:
from IPython.display import display, HTML

def create_comparison_map(r16, r26, g16, g26):
    """2016 vs 2026 インタラクティブ比較マップ"""
    m = folium.Map(
        location=[TAMACHI_LAT, TAMACHI_LNG],
        zoom_start=15,
        tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
        attr='&copy; <a href="https://carto.com/">CARTO</a> &copy; <a href="https://www.openstreetmap.org/copyright">OSM</a>'
    )

    # 800m圏の円
    folium.Circle(
        [TAMACHI_LAT, TAMACHI_LNG],
        radius=RADIUS_M,
        color='gray', fill=False, weight=2,
        popup=f'田町駅から{RADIUS_M}m'
    ).add_to(m)

    # 駅マーカー
    stations = folium.FeatureGroup(name='駅', show=True)
    folium.Marker(
        [35.64564, 139.74753], popup='<b>JR田町駅</b>',
        icon=folium.Icon(color='darkblue', icon='info-sign')
    ).add_to(stations)
    folium.Marker(
        [35.64806, 139.74164], popup='<b>三田駅（都営）</b>',
        icon=folium.Icon(color='darkgreen', icon='info-sign')
    ).add_to(stations)
    stations.add_to(m)

    # --- ラーメン 2016 ---
    fg_r16 = folium.FeatureGroup(name='ラーメン 2016年', show=True)
    for _, row in r16.iterrows():
        popup_html = f"<b>{row['name']}</b><br>評価: {row.get('rating_2016','?')}<br>状態: {row.get('status','')}"
        folium.CircleMarker(
            [row['lat'], row['lng']], radius=7,
            popup=popup_html, tooltip=row['name'],
            color='#FF6600', fill=True, fillColor='#FF6600', fillOpacity=0.7
        ).add_to(fg_r16)
    fg_r16.add_to(m)

    # --- ラーメン 2026 ---
    fg_r26 = folium.FeatureGroup(name='ラーメン 2026年', show=True)
    for _, row in r26.iterrows():
        rating = row.get('rating', '?')
        popup_html = f"<b>{row['name']}</b><br>評価: {rating}<br>住所: {row.get('address','')}"
        folium.CircleMarker(
            [row['lat'], row['lng']], radius=7,
            popup=popup_html, tooltip=row['name'],
            color='#CC0000', fill=True, fillColor='#CC0000', fillOpacity=0.7
        ).add_to(fg_r26)
    fg_r26.add_to(m)

    # --- 牛丼 2016 ---
    fg_g16 = folium.FeatureGroup(name='牛丼 2016年', show=False)
    for _, row in g16.iterrows():
        popup_html = f"<b>{row['name']}</b><br>評価: {row.get('rating_2016','?')}"
        folium.CircleMarker(
            [row['lat'], row['lng']], radius=9,
            popup=popup_html, tooltip=row['name'],
            color='#0066CC', fill=True, fillColor='#0066CC', fillOpacity=0.7
        ).add_to(fg_g16)
    fg_g16.add_to(m)

    # --- 牛丼 2026 ---
    fg_g26 = folium.FeatureGroup(name='牛丼 2026年', show=False)
    for _, row in g26.iterrows():
        rating = row.get('rating', '?')
        popup_html = f"<b>{row['name']}</b><br>評価: {rating}<br>住所: {row.get('address','')}"
        folium.CircleMarker(
            [row['lat'], row['lng']], radius=9,
            popup=popup_html, tooltip=row['name'],
            color='#009933', fill=True, fillColor='#009933', fillOpacity=0.7
        ).add_to(fg_g26)
    fg_g26.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    return m

m = create_comparison_map(r16, r26, g16, g26)

# HTMLファイルとしても保存（ブラウザで直接開ける）
m.save('comparison_map.html')

# 確実にインラインで表示
display(HTML(m._repr_html_()))

## 4. 統計サマリー

In [6]:
print('=' * 50)
print('田町駅から800m以内の店舗数比較')
print('=' * 50)
print(f'{"":>20} {"2016年":>8} {"2026年":>8} {"変化":>8}')
print('-' * 50)
print(f'{"ラーメン":>20} {len(r16):>8} {len(r26):>8} {len(r26)-len(r16):>+8}')
print(f'{"牛丼":>20} {len(g16):>8} {len(g26):>8} {len(g26)-len(g16):>+8}')
print('-' * 50)

# ラーメン店の変遷
common_ramen = set(r16['name']) & set(r26['name'])
only_2016_r = set(r16['name']) - set(r26['name'])
only_2026_r = set(r26['name']) - set(r16['name'])

print(f'\nラーメン店の変遷:')
print(f'  両年に存在: {len(common_ramen)}店')
print(f'  2016年のみ: {len(only_2016_r)}店 (閉店等)')
print(f'  2026年のみ: {len(only_2026_r)}店 (新規開店)')
if common_ramen:
    print(f'  存続店: {", ".join(sorted(common_ramen))}')

# 牛丼店の変遷
common_gyudon = set(g16['name']) & set(g26['name'])
only_2016_g = set(g16['name']) - set(g26['name'])
only_2026_g = set(g26['name']) - set(g16['name'])

print(f'\n牛丼店の変遷:')
print(f'  両年に存在: {len(common_gyudon)}店')
print(f'  2016年のみ: {len(only_2016_g)}店 (閉店等)')
print(f'  2026年のみ: {len(only_2026_g)}店 (新規)')
if common_gyudon:
    print(f'  存続店: {", ".join(sorted(common_gyudon))}')
if only_2016_g:
    print(f'  閉店等: {", ".join(sorted(only_2016_g))}')
if only_2026_g:
    print(f'  新規: {", ".join(sorted(only_2026_g))}')

田町駅から800m以内の店舗数比較
                        2016年    2026年       変化
--------------------------------------------------
                ラーメン       27       30       +3
                  牛丼        5        6       +1
--------------------------------------------------

ラーメン店の変遷:
  両年に存在: 8店
  2016年のみ: 19店 (閉店等)
  2026年のみ: 22店 (新規開店)
  存続店: むらさき山, やっとこ 三田店, ラーメン二郎 三田本店, 博多一瑞亭 三田店, 壱角家 田町店, 横浜家系らーめん 武源家, 香家 三田店, 麻布ラーメン 芝四丁目店

牛丼店の変遷:
  両年に存在: 4店
  2016年のみ: 1店 (閉店等)
  2026年のみ: 2店 (新規)
  存続店: すき家 三田店, すき家 芝四丁目店, なか卯 芝浦店, 松屋 三田店
  閉店等: 吉野家 芝浦店
  新規: すき家 芝浦四丁目店, 吉野家 田町駅芝浦店


## 5. 個別年度マップ

In [7]:
def create_year_map(ramen_df, gyudon_df, year, rating_col='rating_2016'):
    """単一年度のマップ"""
    m = folium.Map(
        location=[TAMACHI_LAT, TAMACHI_LNG],
        zoom_start=15, tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
        attr='&copy; <a href="https://carto.com/">CARTO</a> &copy; <a href="https://www.openstreetmap.org/copyright">OSM</a>'
    )
    folium.Circle(
        [TAMACHI_LAT, TAMACHI_LNG], radius=RADIUS_M,
        color='gray', fill=False, weight=2
    ).add_to(m)

    # ラーメン
    fg_r = folium.FeatureGroup(name=f'ラーメン ({year})', show=True)
    for _, row in ramen_df.iterrows():
        r = row.get(rating_col, row.get('rating', '?'))
        folium.CircleMarker(
            [row['lat'], row['lng']], radius=8,
            popup=f"<b>{row['name']}</b><br>評価: {r}",
            tooltip=row['name'],
            color='#FF6600', fill=True, fillColor='#FF6600', fillOpacity=0.8
        ).add_to(fg_r)
    fg_r.add_to(m)

    # 牛丼
    fg_g = folium.FeatureGroup(name=f'牛丼 ({year})', show=True)
    for _, row in gyudon_df.iterrows():
        r = row.get(rating_col, row.get('rating', '?'))
        folium.CircleMarker(
            [row['lat'], row['lng']], radius=10,
            popup=f"<b>{row['name']}</b><br>評価: {r}",
            tooltip=row['name'],
            color='#0066CC', fill=True, fillColor='#0066CC', fillOpacity=0.8
        ).add_to(fg_g)
    fg_g.add_to(m)

    folium.LayerControl().add_to(m)
    return m

print('=== 2016年 ===')
m2016 = create_year_map(r16, g16, '2016', 'rating_2016')
m2016.save('map_2016.html')
display(HTML(m2016._repr_html_()))

=== 2016年 ===


In [8]:
print('=== 2026年 ===')
m2026 = create_year_map(r26, g26, '2026', 'rating')
m2026.save('map_2026.html')
display(HTML(m2026._repr_html_()))

=== 2026年 ===


## 技術メモ

### 2016年データの座標抽出パイプライン
1. PDFから地図画像を切り出し（`pdf2image`）
2. OpenCVでHSV色空間によるオレンジピン検出
3. 輪郭検出 + 形状フィルタ（面積・コンパクト度）
4. 4参照点（三田駅・田町駅西口・JR田町・三田通り交差前）でアフィン変換
5. 座標精度: RMS ≈ 15m
6. 食べログ閉店ページから住所・座標を補完

### 2026年データ
- 食べログ一覧ページからスクレイピング
- 各店舗詳細ページのHTML内に`lat`, `lng`が直接記載
- `requests` + `BeautifulSoup`で取得